# Lecture 2 · Notebook 1 — From bytes to tokens
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture02/lecture02_1_from_bytes_to_tokens.ipynb)

**Companion to [Chapters 2–3 of the video lecture](https://nlp-course-cuhksz.github.io/lecture02.html#p2-s36)** (slides 34–61). Before a model can learn anything about words,
the text has to be stored as bytes, decoded back into characters, and cut into tokens. Every one of those steps involves a choice.
In this notebook you will:

1. look at the **code points and bytes** behind 自然语言处理, and produce mojibake on purpose,
2. see why "one visible character" is not always one code point, and what Unicode **normalization** does,
3. count **tokens vs. types** in a novel and draw **Zipf's long tail**,
4. watch a Chinese segmenter split **乒乓球拍卖完了** two ways,
5. implement the **BPE token learner and segmenter** from scratch, and
6. compare GPT-2's byte-level BPE with a newer multilingual tokenizer on English and Chinese.

Runs on CPU in about 2 minutes (downloads: one public-domain novel and two tokenizers, a few MB in total).

In [ ]:
!pip -q install jieba

In [ ]:
import re, unicodedata, collections, requests
import numpy as np
import matplotlib.pyplot as plt

## 1 · Code points and bytes
Unicode gives every character a **code point** (an ID such as `U+0041` for `A`). An **encoding** decides how that ID is written as bytes:
UTF-8 uses 1–4 bytes, UTF-16 uses 2 or 4, and the older Chinese standard GBK uses 2 bytes per Chinese character (slides 34–38).

In [ ]:
text = "自然语言处理"
print(f"{'char':<5}{'code point':<12}{'UTF-8':<12}{'UTF-16-BE':<12}{'GBK'}")
for ch in text:
    print(f"{ch:<4} U+{ord(ch):04X}     {ch.encode('utf-8').hex(' '):<12}{ch.encode('utf-16-be').hex(' '):<12}{ch.encode('gbk').hex(' ')}")

print()
for s in ["A", "é", "中", "😀"]:
    print(f"{s!r}: code point U+{ord(s):04X}, UTF-8 {len(s.encode('utf-8'))} bytes, UTF-16 {len(s.encode('utf-16-be'))} bytes")

The same character, three different byte sequences. If the reader assumes the wrong encoding, you get **mojibake** (乱码) —
the "strange symbols" from slide 35:

In [ ]:
gbk_bytes = text.encode("gbk")
print("GBK bytes read as UTF-8 :", gbk_bytes.decode("utf-8", errors="replace"))
print("UTF-8 bytes read as GBK :", text.encode("utf-8").decode("gbk", errors="replace"))
print("UTF-8 bytes read as Latin-1:", text.encode("utf-8").decode("latin-1"))
print("correct round trip      :", gbk_bytes.decode("gbk"))

## 2 · Visible characters ≠ code points, and normalization
A symbol on screen can be several code points (slide 37). Two strings that *look* identical can compare as different — a classic source of
bugs in search and deduplication. Unicode **normalization** (NFC, NFKC) maps such variants to one form.

In [ ]:
a = "café"            # é as one code point U+00E9
b = "cafe\u0301"      # e + COMBINING ACUTE ACCENT
print(a, b, "| equal?", a == b, "| lengths", len(a), len(b))
print("after NFC:", unicodedata.normalize("NFC", a) == unicodedata.normalize("NFC", b))

family = "👨\u200d👩\u200d👧"   # one emoji on screen
print(family, "->", len(family), "code points:", [f"U+{ord(c):04X}" for c in family], "|", len(family.encode("utf-8")), "UTF-8 bytes")

NFKC goes further and folds *compatibility* variants: full-width letters (common in Chinese text), circled digits, and ligatures.
The slide titles of this very lecture contain a ligature — *"Words aren’t just deﬁned by blanks"* uses `ﬁ` (U+FB01), copied from a PDF.
A search for "defined" would miss it.

In [ ]:
for s in ["ＮＬＰ　２０２６", "deﬁned", "①②③", "ｶﾀｶﾅ"]:
    print(f"{s!r:22} -> {unicodedata.normalize('NFKC', s)!r}")
title = "Words aren’t just deﬁned by blanks"
print("\n'defined' found in the slide title?  before NFKC:", "defined" in title, "| after NFKC:", "defined" in unicodedata.normalize("NFKC", title))

## 3 · Tokens, types and Zipf's law
Download *Pride and Prejudice* from Project Gutenberg, lower-case it and split it into word tokens with a simple regular expression.
**Tokens** are running words; **types** are distinct words (slide 45).

In [ ]:
import os
if not os.path.exists("pride_and_prejudice.txt"):     # download once, reuse on re-runs
    open("pride_and_prejudice.txt", "w", encoding="utf-8").write(
        requests.get("https://www.gutenberg.org/cache/epub/1342/pg1342.txt", timeout=120).text)
raw = open("pride_and_prejudice.txt", encoding="utf-8").read()
start = raw.find("*** START"); end = raw.find("*** END")
book = raw[raw.find("\n", start) + 1:end]
tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", book.lower())
counts = collections.Counter(tokens)
print(f"{len(tokens):,} tokens, {len(counts):,} types")
print("most common:", counts.most_common(12))
hapax = [w for w, c in counts.items() if c == 1]
print(f"{len(hapax):,} types ({len(hapax)/len(counts):.0%}) occur exactly once, e.g.", hapax[:8])

Zipf's law says frequency falls roughly as 1/rank: a straight line on a log–log plot (slide 46). The right panel shows the other side of
the long tail: as you read more text you **keep meeting new types**, so no fixed word list ever covers everything.

In [ ]:
freqs = np.array(sorted(counts.values(), reverse=True))
ranks = np.arange(1, len(freqs) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].loglog(ranks, freqs, ".", ms=3, label="Pride and Prejudice")
ax[0].loglog(ranks, freqs[0] / ranks, "r--", label="C / rank")
ax[0].set_xlabel("rank"); ax[0].set_ylabel("frequency"); ax[0].legend(); ax[0].set_title("Zipf's law")
seen, growth = set(), []
for i, t in enumerate(tokens, 1):
    seen.add(t)
    if i % 1000 == 0: growth.append((i, len(seen)))
g = np.array(growth)
ax[1].plot(g[:, 0], g[:, 1]); ax[1].set_xlabel("tokens read"); ax[1].set_ylabel("distinct types seen"); ax[1].set_title("New words keep coming")
plt.tight_layout(); plt.show()

top100 = freqs[:100].sum() / freqs.sum()
print(f"The 100 most frequent types cover {top100:.0%} of all tokens; half of all types appear at most {int(np.median(freqs))} time(s).")

**Word mapping** (slide 48): lower-casing, stemming and lemmatization shrink the vocabulary, but each choice throws information away.
A Porter stemmer is fast but crude:

In [ ]:
from nltk.stem import PorterStemmer
stem = PorterStemmer().stem
for w in ["cats", "running", "went", "studies", "organize", "organization", "university", "universe"]:
    print(f"{w:13} -> {stem(w)}")
print("\nLower-casing merges 'US' (the country) and 'us' (the pronoun):", "US".lower() == "us")

## 4 · Chinese: where are the word boundaries?
Chinese has no spaces. The segmenter `jieba` picks one segmentation by default; its *full mode* lists every dictionary word it can find,
which exposes the ambiguity from slide 56: 球拍 (paddle) + 卖完 (sold out) vs. 拍卖 (auction).

In [ ]:
import jieba, logging
jieba.setLogLevel(logging.WARNING)
for s in ["乒乓球拍卖完了", "南京市长江大桥", "我们一起商量商量"]:
    print(s)
    print("   default  :", " / ".join(jieba.lcut(s)))
    print("   full mode:", " / ".join(jieba.lcut(s, cut_all=True)))

## 5 · Byte Pair Encoding, from scratch
BPE has two parts (slide 60). The **token learner** starts from characters and repeatedly merges the most frequent adjacent pair.
The **token segmenter** applies the learned merges, in order, to new words. `_` marks the end of a word.

In [ ]:
def learn_bpe(word_counts, num_merges, verbose=False):
    vocab = {tuple(w) + ("_",): c for w, c in word_counts.items()}
    merges = []
    for step in range(num_merges):
        pairs = collections.Counter()
        for sym, c in vocab.items():
            for a, b in zip(sym, sym[1:]):
                pairs[a, b] += c
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        merges.append(best)
        new_vocab = {}
        for sym, c in vocab.items():
            if best[0] not in sym:                 # quick skip: the pair cannot occur in this word
                new_vocab[sym] = c; continue
            out, i = [], 0
            while i < len(sym):
                if i < len(sym) - 1 and (sym[i], sym[i + 1]) == best:
                    out.append(sym[i] + sym[i + 1]); i += 2
                else:
                    out.append(sym[i]); i += 1
            new_vocab[tuple(out)] = c
        vocab = new_vocab
        if verbose:
            print(f"merge {step+1:2d}: {best[0]!r:6} + {best[1]!r:6} (count {pairs[best]:2d})  ->", " ".join(" ".join(s) for s in vocab))
    return merges

def segment(word, merges):
    sym = list(word) + ["_"]
    for a, b in merges:
        i, out = 0, []
        while i < len(sym):
            if i < len(sym) - 1 and sym[i] == a and sym[i + 1] == b:
                out.append(a + b); i += 2
            else:
                out.append(sym[i]); i += 1
        sym = out
    return sym

toy = {"low": 5, "lowest": 2, "newer": 6, "wider": 3, "new": 2}   # the toy corpus from Jurafsky & Martin, ch. 2
toy_merges = learn_bpe(toy, 8, verbose=True)
for w in ["lower", "newest", "widest", "slow"]:
    print(f"{w:8} -> {segment(w, toy_merges)}")

`lower` never occurred in the toy corpus, yet it is split into two learned pieces, `low` + `er_`. `newest` and `widest` reuse what they
can (`new`) and fall back to single letters for the rest. Either way nothing is unknown: BPE solves the **out-of-vocabulary** problem of
slide 49 by falling back to smaller units. Now learn 1,000 merges from the whole novel (takes a few seconds):

In [ ]:
merges = learn_bpe(counts, 1000)
print("first 20 merges:", ["".join(m) for m in merges[:20]])
for w in ["prejudice", "unhappiness", "disgracefully", "tokenization", "darcy's", "chatgpt"]:
    print(f"{w:14} -> {' '.join(segment(w, merges))}")

Frequent words become one token; rare or new words split into pieces that are *sometimes* morphemes (`un`, `ness`, `ly`) and sometimes
arbitrary substrings — BPE follows frequency, not linguistics (slide 60).

## 6 · Real tokenizers: GPT-2 vs. Qwen2.5
GPT-2 uses **byte-level BPE** with 50,257 tokens (slide 61): every input is first turned into UTF-8 bytes, so nothing is ever unknown.
Qwen2.5 uses the same idea with a much larger vocabulary trained on far more multilingual text. Let's compare them on the sentences from
slide 58. Only the small tokenizer files are downloaded, not the models.

In [ ]:
from transformers import AutoTokenizer
toks = {"GPT-2": AutoTokenizer.from_pretrained("gpt2"),
        "Qwen2.5": AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")}
for name, tok in toks.items():
    print(f"{name}: vocabulary size {len(tok):,}")

en = "A helpful rule of thumb is that one token generally corresponds to ~4 characters of text for common English text."
zh = "一个有用的经验法则是，对于通常英文文本而言，一个标记对应约4个字符的文本。"
for label, s in [("English", en), ("Chinese", zh)]:
    for name, tok in toks.items():
        ids = tok.encode(s)
        print(f"{label:8}{name:8} {len(ids):3d} tokens for {len(s):3d} characters = {len(s)/len(ids):.2f} chars/token")

In [ ]:
def show(tok, s):
    ids = tok.encode(s)
    return " | ".join(tok.decode([i]) for i in ids)   # a piece that is only part of a character's bytes prints as �

for s in ["unbelievably", " Shenzhen", "自然语言处理", "😀"]:
    for name, tok in toks.items():
        print(f"{name:8} {s!r:16} -> {show(tok, s)}")
    print()
print("UTF-8 bytes of 自:", "自".encode("utf-8"), "-> GPT-2 tokens:", toks["GPT-2"].convert_ids_to_tokens(toks["GPT-2"].encode("自")))

GPT-2 prints `�` for many Chinese pieces: those tokens are **single bytes or byte pairs**, only part of a character's 3-byte UTF-8 code
(the last line shows its tokens for 自, displayed with GPT-2's printable stand-ins for raw bytes). It can represent anything, but Chinese
costs it about two tokens per character. Qwen2.5's tokenizer, trained on far more Chinese text, has whole words such as 自然, 语言 and
处理 as single tokens, so the same sentence becomes much shorter. As slide 61
says: being able to *represent* a string is not the same as *understanding* it.

## Exercises
1. Type your own name (in English and in Chinese) into both tokenizers. How many tokens does each need? Why might this matter for the
   cost and quality of an LLM in your language?
2. Change the number of BPE merges learned from the novel (200, 1,000, 3,000). How do the segmentations of `unhappiness` and
   `tokenization` change? At what point do whole words appear?
3. Fit a straight line to the log–log Zipf plot (`np.polyfit(np.log(ranks), np.log(freqs), 1)`). Is the slope close to −1?
4. Find another Chinese sentence where the default `jieba` segmentation is wrong. Does adding context fix it?

**Next:** [Notebook 2 — from counts to vectors]({colab(N2)})